# Large-Scale Portfolio Briefs Generation V2

This notebook runs Brief V2 batches for a portfolio of `RP_ENTITY_ID` values, monitors each batch to completion, retrieves generated bullets, previews the first 10 entities, and exports all bullet results to Excel.

## Step 1: Load Company Identifiers from CSV File

Read the portfolio CSV, locate the `RP_ENTITY_ID` column, remove empty or duplicate identifiers, and prepare the entity list used by the V2 batch APIs.

In [1]:
from __future__ import annotations

import json
import os
import time
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import Markdown, display

JsonDict = dict[str, Any]

CSV_PATH = Path("static/data/US_100.csv")
df = pd.read_csv(CSV_PATH, dtype=str)

entity_id_column = next((c for c in df.columns if c.strip().upper() == "RP_ENTITY_ID"), None)
if entity_id_column is None:
    raise ValueError(f"RP_ENTITY_ID column not found in {CSV_PATH}")

ids: list[str] = (
    df[entity_id_column]
    .astype(str)
    .str.strip()
    .replace({"": None})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Loaded {len(ids)} unique RP_ENTITY_ID values from {CSV_PATH}")

Loaded 107 unique RP_ENTITY_ID values from static/data/US_100.csv


## Step 2: Configure Brief V2 Batch Processing

Configure the V2 API endpoints, batch size, and report window. The V2 workflow uses the selected `2026-02-01T00:00:00` to `2026-03-30T23:59:59` window.

In [2]:
# Batch size follows the V1 notebook's production-oriented default.
BATCH_SIZE: int = 50
companies: list[str] = ids

API_BASE_URL: str = "http://localhost:8000"
RUN_PARALLEL_URL: str = f"{API_BASE_URL}/api/v1/batch/run-parallel"
BULLETS_URL: str = f"{API_BASE_URL}/api/v1/batch/bullets"


FORCE_WINDOW_START: str = "2026-02-01T00:00:00"
FORCE_WINDOW_END: str = "2026-03-30T23:59:59"

POLL_INTERVAL_SECONDS: int = 30
BATCH_TIMEOUT_SECONDS: int = 3600

# Optional token support, retained from the V1 workflow if the local service requires it.
token = os.environ.get("API_TOKEN") or os.environ.get("TOKEN") or os.environ.get("API_KEY")
params: dict[str, str] = {"token": token} if token else {}

print(f"Using {len(companies)} companies")
print(f"Batch size: {BATCH_SIZE}")
print(f"Report window: {FORCE_WINDOW_START} to {FORCE_WINDOW_END}")

Using 107 companies
Batch size: 50
Report window: 2026-02-01T00:00:00 to 2026-03-30T23:59:59


## Step 3: Set Output Folder and File Names

All V2 artifacts are written under `output/` with V2-specific file names so they do not overwrite V1 outputs.

In [17]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRIEF_V2_BATCH_SUMMARY_FILE = OUTPUT_DIR / "brief_v2_batch_summaries.json"
BRIEF_V2_FIRST5_BULLETS_FILE = OUTPUT_DIR / "brief_v2_first5_bullets.json"
BRIEF_V2_ALL_BULLETS_FILE = OUTPUT_DIR / "brief_v2_all_bullets.json"
OUT_XLSX = OUTPUT_DIR / "portfolio_briefs_generation_v2.xlsx"

## Step 4: Define API and Formatting Helpers

These helpers keep the V2 API calls, polling logic, JSON persistence, and citation formatting reusable across the notebook.

In [4]:
def chunk_entities(entity_ids: list[str], batch_size: int) -> list[list[str]]:
    """Split entity identifiers into fixed-size batches."""
    return [entity_ids[start : start + batch_size] for start in range(0, len(entity_ids), batch_size)]


def request_json(method: str, url: str, *, timeout: int = 60, **kwargs: Any) -> JsonDict:
    """Call the local Brief V2 API and return parsed JSON with useful HTTP error context."""
    response = requests.request(method, url, params=params, timeout=timeout, **kwargs)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        message = f"{method} {url} failed with {response.status_code}: {response.text}"
        raise requests.HTTPError(message) from exc
    return response.json()


def status_url_for(batch_id: str) -> str:
    """Build the V2 batch status endpoint for a submitted batch."""
    return f"{API_BASE_URL}/api/v1/batch/parallel/{batch_id}/status"


def is_batch_complete(status_payload: JsonDict) -> bool:
    """Return True when all entities in a V2 batch have either succeeded or failed."""
    total = int(status_payload.get("total") or 0)
    succeeded = int(status_payload.get("succeeded") or 0)
    failed = int(status_payload.get("failed") or 0)
    running = int(status_payload.get("running") or 0)
    not_started = int(status_payload.get("not_started") or 0)

    if total > 0:
        return succeeded + failed >= total
    return running == 0 and not_started == 0 and succeeded + failed > 0


def status_summary(status_payload: JsonDict) -> str:
    """Format a compact status line for notebook progress output."""
    return (
        f"succeeded={status_payload.get('succeeded', 0)}, "
        f"failed={status_payload.get('failed', 0)}, "
        f"running={status_payload.get('running', 0)}, "
        f"not_started={status_payload.get('not_started', 0)}, "
        f"total={status_payload.get('total', 0)}"
    )


def save_json(path: Path, payload: Any) -> None:
    """Persist payload as indented UTF-8 JSON."""
    with path.open("w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, ensure_ascii=False, indent=2)


def fetch_bullets_for_entities(entity_ids: list[str]) -> JsonDict:
    """Retrieve saved V2 bullets for a list of entities."""
    if not entity_ids:
        return {"results": [], "total_entities": 0, "total_bullets": 0}
    return request_json("POST", BULLETS_URL, json={"entity_ids": entity_ids}, timeout=180)

## Step 5: Run Brief V2 Batches Sequentially

Submit one `run-parallel` batch at a time, poll its status until every entity has either succeeded or failed, then move to the next batch. This matches the requested flow: once a batch is complete, start the next batch.

In [5]:
batch_summaries: list[JsonDict] = []
entity_batches = chunk_entities(companies, BATCH_SIZE)

print(f"Batch starting date and time: {datetime.now()}")

for batch_number, batch in enumerate(entity_batches, start=1):
    batch_start = (batch_number - 1) * BATCH_SIZE + 1
    batch_end = batch_start + len(batch) - 1
    payload_batch: JsonDict = {
        "entity_ids": batch,
        "force_window_start": FORCE_WINDOW_START,
        "force_window_end": FORCE_WINDOW_END,
    }

    summary: JsonDict = {
        "batch_number": batch_number,
        "batch_start": batch_start,
        "batch_end": batch_end,
        "entity_count": len(batch),
        "entity_ids": batch,
        "payload": payload_batch,
        "status": "submitted",
        "submitted_at_local": datetime.now().isoformat(),
    }

    try:
        print(f"Submitting batch {batch_start}-{batch_end} ({len(batch)} entities)...")
        submit_response = request_json("POST", RUN_PARALLEL_URL, json=payload_batch, timeout=180)
        summary["submission_response"] = submit_response
    except requests.RequestException as exc:
        print(f"Batch {batch_start}-{batch_end} submission failed: {exc}")
        traceback.print_exc()
        summary["status"] = "submit_failed"
        summary["error"] = str(exc)
        batch_summaries.append(summary)
        continue

    batch_id = str(submit_response.get("batch_id") or "").strip()
    if not batch_id:
        print(f"Batch {batch_start}-{batch_end} did not return a batch_id; skipping status polling.")
        summary["status"] = "missing_batch_id"
        batch_summaries.append(summary)
        continue

    summary["batch_id"] = batch_id
    summary["submitted_at"] = submit_response.get("submitted_at")
    summary["total"] = submit_response.get("total")

    final_status_response: JsonDict | None = None
    waited_seconds = 0

    while waited_seconds <= BATCH_TIMEOUT_SECONDS:
        try:
            status_response = request_json("GET", status_url_for(batch_id), timeout=60)
            final_status_response = status_response
            print(f"Batch {batch_start}-{batch_end} status: {status_summary(status_response)}")

            if is_batch_complete(status_response):
                failed = int(status_response.get("failed") or 0)
                summary["status"] = "completed_with_failures" if failed else "completed"
                break
        except requests.RequestException as exc:
            print(f"Status check error for batch {batch_id}: {exc}")
            summary.setdefault("status_errors", []).append(str(exc))

        time.sleep(POLL_INTERVAL_SECONDS)
        waited_seconds += POLL_INTERVAL_SECONDS

    if final_status_response is None:
        summary["status"] = "status_unavailable"
    elif not is_batch_complete(final_status_response):
        summary["status"] = "timeout"

    summary["waited_seconds"] = waited_seconds
    summary["completed_at_local"] = datetime.now().isoformat()
    summary["final_status_response"] = final_status_response
    batch_summaries.append(summary)

    print(f"Batch {batch_start}-{batch_end} finished with status: {summary['status']}")

save_json(BRIEF_V2_BATCH_SUMMARY_FILE, batch_summaries)

print(f"Saved {len(batch_summaries)} batch summaries to {BRIEF_V2_BATCH_SUMMARY_FILE}")
print(f"Batch completion date and time: {datetime.now()}")

Batch starting date and time: 2026-04-25 17:09:48.060164
Submitting batch 1-50 (50 entities)...
Batch 1-50 status: succeeded=0, failed=0, running=0, not_started=50, total=50
Batch 1-50 status: succeeded=0, failed=0, running=10, not_started=40, total=50
Batch 1-50 status: succeeded=0, failed=0, running=10, not_started=40, total=50
Batch 1-50 status: succeeded=2, failed=0, running=8, not_started=40, total=50
Batch 1-50 status: succeeded=6, failed=0, running=6, not_started=38, total=50
Batch 1-50 status: succeeded=9, failed=0, running=6, not_started=35, total=50
Batch 1-50 status: succeeded=11, failed=0, running=6, not_started=33, total=50
Batch 1-50 status: succeeded=12, failed=0, running=7, not_started=31, total=50
Batch 1-50 status: succeeded=15, failed=0, running=6, not_started=29, total=50
Batch 1-50 status: succeeded=18, failed=0, running=4, not_started=28, total=50
Batch 1-50 status: succeeded=21, failed=0, running=4, not_started=25, total=50
Batch 1-50 status: succeeded=21, failed

## Step 6: Retrieve Bullets for the First 5 Entities

After batches complete, fetch bullets for the first 5 entities and save the raw response for inspection.

In [18]:
preview_entity_ids: list[str] = companies[:5]
first5_bullets_response = fetch_bullets_for_entities(preview_entity_ids)
save_json(BRIEF_V2_FIRST5_BULLETS_FILE, first5_bullets_response)

first5_results: list[JsonDict] = first5_bullets_response.get("results", []) or []

print(
    f"Retrieved bullets for {len(first5_results)} preview entities "
    f"with {first5_bullets_response.get('total_bullets', 0)} total bullets."
)
print(f"Saved preview bullets to {BRIEF_V2_FIRST5_BULLETS_FILE}")

Retrieved bullets for 5 preview entities with 26 total bullets.
Saved preview bullets to output/brief_v2_first5_bullets.json


## Step 7: Display First 5 Entity Bullet Results

Render the first 5 V2 bullet results in a notebook-friendly analyst view, including run metadata, bullet text, citations, and discard summaries.

In [21]:
def source_names_from_citation(citation: JsonDict) -> list[str]:
    """Extract citation source names from V2 source_references metadata."""
    source_references = citation.get("source_references") or []
    source_names: list[str] = []

    if isinstance(source_references, list):
        for source_reference in source_references:
            if not isinstance(source_reference, dict):
                continue
            source_name = str(source_reference.get("source_name") or "").strip()
            if source_name and source_name not in source_names:
                source_names.append(source_name)

    fallback_source_name = str(citation.get("source_name") or "").strip()
    if fallback_source_name and fallback_source_name not in source_names:
        source_names.append(fallback_source_name)

    return source_names


def citations_table(citations: list[JsonDict]) -> pd.DataFrame:
    """Build a readable citation table for notebook preview output."""
    rows: list[JsonDict] = []

    for citation_number, citation in enumerate(citations, start=1):
        source_names = source_names_from_citation(citation)
        rows.append({
            "#": citation_number,
            "source": ", ".join(source_names),
            "headline": str(citation.get("headline") or "").strip(),
            "citation_id": str(citation.get("id") or "").strip(),
        })

    return pd.DataFrame(rows, columns=["#", "source", "headline", "citation_id"])


def count_items(value: Any) -> int:
    """Count list-like discard buckets from the V2 response."""
    return len(value) if isinstance(value, list) else 0


def render_v2_entity_result(entity_result: JsonDict, *, top_n: int | None = None) -> None:
    """Display one V2 entity result with runs, bullets, citations, and discard counts."""
    entity_id = entity_result.get("entity_id", "N/A")
    entity_name = entity_result.get("entity_name") or "Unknown"
    found = entity_result.get("found")
    total_runs = entity_result.get("total_runs", 0)
    total_bullets = entity_result.get("total_bullets", 0)

    display(Markdown(
        f"## {entity_name}\n"
        f"**RP_ENTITY_ID:** `{entity_id}`  |  **Found:** {found}  |  "
        f"**Runs:** {total_runs}  |  **Bullets:** {total_bullets}"
    ))

    runs = entity_result.get("runs", []) or []
    if not runs:
        display(Markdown("_No runs found for this entity._"))
        return

    for run_index, run in enumerate(runs, start=1):
        bullets = run.get("bullets", []) or []
        display(Markdown(
            f"### Run {run_index}: `{run.get('run_id', 'N/A')}`\n"
            f"**Window:** {run.get('report_window_start', 'N/A')} to {run.get('report_window_end', 'N/A')}  |  "
            f"**Created:** {run.get('run_created_at', 'N/A')}  |  "
            f"**Saved:** {run.get('bullets_saved', len(bullets))}  |  "
            f"**Discarded:** {run.get('bullets_discarded', 0)}"
        ))

        discard_summary = (
            f"**Discarded by relevance:** {count_items(run.get('discarded_by_relevance'))}  |  "
            f"**Discarded by grounding:** {count_items(run.get('discarded_by_grounding'))}  |  "
            f"**Discarded by novelty:** {count_items(run.get('discarded_by_novelty'))}"
        )
        display(Markdown(discard_summary))

        if not bullets:
            display(Markdown("_No bullet points found for this run._"))
            continue

        limit = top_n if top_n is not None else len(bullets)
        table_rows: list[JsonDict] = []

        for bullet_number, bullet in enumerate(bullets[:limit], start=1):
            text = str(bullet.get("text") or "").strip()
            citations = bullet.get("citations", []) or []
            citation_df = citations_table(citations)
            decision_line = (
                f"embedding_decision={bullet.get('embedding_decision', '')}, "
                f"search_action={bullet.get('search_action', '')}, "
                f"not_fully_novel={bullet.get('not_fully_novel', '')}"
            )

            display(Markdown(f"{bullet_number}. {text}\n\n**Decisions:** `{decision_line}`"))
            display(Markdown("**Citations:**"))
            if citation_df.empty:
                display(Markdown("_No citations found for this bullet._"))
            else:
                display(citation_df)

            table_rows.append({
                "trace_id": bullet.get("trace_id", ""),
                "bullet": text,
                "citation_count": len(citations),
                "citation_ids": "; ".join(str(citation.get("id") or "") for citation in citations),
                "embedding_decision": bullet.get("embedding_decision", ""),
                "search_action": bullet.get("search_action", ""),
                "not_fully_novel": bullet.get("not_fully_novel", ""),
            })

        display(Markdown("**Raw table (for copy/export):**"))
        display(pd.DataFrame(table_rows))


for preview_result in first5_results:
    render_v2_entity_result(preview_result)

## Costco Wholesale Corp.
**RP_ENTITY_ID:** `B8EF97`  |  **Found:** True  |  **Runs:** 1  |  **Bullets:** 5

### Run 1: `3e5d177b-931c-481e-88b2-f8887cf9f491`
**Window:** 2026-02-01T00:00:00 to 2026-03-30T23:59:59  |  **Created:** 2026-04-25T21:11:32.236752  |  **Saved:** 5  |  **Discarded:** 19

**Discarded by relevance:** 3  |  **Discarded by grounding:** 3  |  **Discarded by novelty:** 13

1. Costco Wholesale Corp. reported fiscal Q2 2026 earnings per share of $4.58, up 13.9% from $4.02 a year earlier, beating analyst consensus estimates.

**Decisions:** `embedding_decision=keep, search_action=keep, not_fully_novel=False`

**Citations:**

...

6. U.S. Bancorp Co. rolled out a connected wealth model for emerging affluent investors, combining hybrid human advice, self-directed investing, and integrated banking data to simplify the investing experience and provide flexible engagement options.

**Decisions:** `embedding_decision=keep, search_action=keep, not_fully_novel=False`

**Citations:**

,#,source,headline,citation_id
0,1,,US Bancorp Advisors targets emerging affluent ...,CQS:B454294FAE97EFC8256000DC21D4918D-1
1,2,,US Bancorp Advisors targets emerging affluent ...,CQS:B454294FAE97EFC8256000DC21D4918D-2
2,3,,US Bancorp Advisors targets emerging affluent ...,CQS:B454294FAE97EFC8256000DC21D4918D-3


**Raw table (for copy/export):**

,trace_id,bullet,citation_count,citation_ids,embedding_decision,search_action,not_fully_novel
0,c9acf465-33e2-4522-b86e-3331845fc6fe,U.S. Bancorp Co. announced that Toby Clements ...,6,CQS:C276C0AAD052B7E8E70AEFA7A3763667-1; CQS:DA...,keep,keep,False
1,58fa47df-e32b-444d-b54e-5a90cdc7f070,U.S. Bancorp Co. appointed Alan Flanagan as he...,8,CQS:153021EE2799DC167732FD18DB58D910-1; CQS:15...,keep,keep,False
2,b9c7b49d-8859-47d4-b031-d87ece464e8b,U.S. Bancorp Co. named Ryan K. Nelson as Presi...,2,CQS:1BEAE60F29A6C3070FC7602F017AF061-3; CQS:1B...,keep,keep,False
3,e8b6bf1a-f60e-4e22-ba0e-a3aa39eff7bf,U.S. Bancorp Co. introduced new six- and seven...,1,CQS:4ECE55827431EB356403E91864A1228D-1,keep,keep,False
4,3b58a896-2318-44d6-948a-366d6cafb779,U.S. Bancorp Co. launched a suite of wealth ma...,6,CQS:1BEAE60F29A6C3070FC7602F017AF061-1; CQS:1B...,keep,keep,False
5,1cc84fd7-2785-4354-ad9f-e49dfbc28fb9,U.S. Bancorp Co. rolled out a connected wealth...,3,CQS:B454294FAE97EFC8256000DC21D4918D-1; CQS:B4...,keep,keep,False


## Step 8: Retrieve All Bullets and Export to Excel

Fetch saved bullets for all entities in the same `BATCH_SIZE` chunks, persist the raw combined V2 response, flatten runs/bullets/citations, and export the results to Excel.

In [20]:
def source_names_from_citation_for_export(citation: JsonDict) -> list[str]:
    """Extract citation source names from V2 source_references metadata."""
    source_references = citation.get("source_references") or []
    source_names: list[str] = []

    if isinstance(source_references, list):
        for source_reference in source_references:
            if not isinstance(source_reference, dict):
                continue
            source_name = str(source_reference.get("source_name") or "").strip()
            if source_name and source_name not in source_names:
                source_names.append(source_name)

    fallback_source_name = str(citation.get("source_name") or "").strip()
    if fallback_source_name and fallback_source_name not in source_names:
        source_names.append(fallback_source_name)

    return source_names


def citation_fields(citations: list[JsonDict]) -> JsonDict:
    """Flatten citation objects into semicolon-delimited Excel fields."""
    return {
        "citation_ids": "; ".join(str(citation.get("id") or "") for citation in citations),
        "citation_headlines": "; ".join(str(citation.get("headline") or "") for citation in citations),
        "citation_sources": "; ".join(
            ", ".join(source_names_from_citation_for_export(citation)) for citation in citations
        ),
        "citation_texts": "\n---\n".join(str(citation.get("text") or "") for citation in citations),
    }


def entity_run_base_row(entity_result: JsonDict, run: JsonDict | None = None) -> JsonDict:
    """Build common export columns for an entity/run pair."""
    run_payload = run or {}
    return {
        "rp_entity_id": entity_result.get("entity_id", ""),
        "entity_name": entity_result.get("entity_name", ""),
        "found": entity_result.get("found", ""),
        "total_runs": entity_result.get("total_runs", 0),
        "total_bullets": entity_result.get("total_bullets", 0),
        "run_id": run_payload.get("run_id", ""),
        "report_window_start": run_payload.get("report_window_start", ""),
        "report_window_end": run_payload.get("report_window_end", ""),
        "run_created_at": run_payload.get("run_created_at", ""),
        "run_bullet_count": run_payload.get("bullet_count", 0),
        "bullets_saved": run_payload.get("bullets_saved", 0),
        "bullets_discarded": run_payload.get("bullets_discarded", 0),
        "discarded_by_relevance_count": count_items(run_payload.get("discarded_by_relevance")),
        "discarded_by_grounding_count": count_items(run_payload.get("discarded_by_grounding")),
        "discarded_by_novelty_count": count_items(run_payload.get("discarded_by_novelty")),
    }


def flatten_v2_results(results: list[JsonDict]) -> pd.DataFrame:
    """Flatten V2 bullet results into one Excel-friendly row per bullet."""
    rows: list[JsonDict] = []

    for entity_result in results:
        runs = entity_result.get("runs", []) or []
        if not runs:
            rows.append({
                **entity_run_base_row(entity_result),
                "bullet_number": "",
                "trace_id": "",
                "bullet_text": "",
                "embedding_decision": "",
                "search_action": "",
                "not_fully_novel": "",
                "citation_count": 0,
                **citation_fields([]),
            })
            continue

        for run in runs:
            bullets = run.get("bullets", []) or []
            if not bullets:
                rows.append({
                    **entity_run_base_row(entity_result, run),
                    "bullet_number": "",
                    "trace_id": "",
                    "bullet_text": "",
                    "embedding_decision": "",
                    "search_action": "",
                    "not_fully_novel": "",
                    "citation_count": 0,
                    **citation_fields([]),
                })
                continue

            for bullet_number, bullet in enumerate(bullets, start=1):
                citations = bullet.get("citations", []) or []
                rows.append({
                    **entity_run_base_row(entity_result, run),
                    "bullet_number": bullet_number,
                    "trace_id": bullet.get("trace_id", ""),
                    "bullet_text": str(bullet.get("text") or "").strip(),
                    "embedding_decision": bullet.get("embedding_decision", ""),
                    "search_action": bullet.get("search_action", ""),
                    "not_fully_novel": bullet.get("not_fully_novel", ""),
                    "citation_count": len(citations),
                    **citation_fields(citations),
                })

    return pd.DataFrame(rows)


all_bullets_results: list[JsonDict] = []
all_bullets_responses: list[JsonDict] = []

for batch_number, batch in enumerate(chunk_entities(companies, BATCH_SIZE), start=1):
    batch_start = (batch_number - 1) * BATCH_SIZE + 1
    batch_end = batch_start + len(batch) - 1
    print(f"Retrieving bullets for entities {batch_start}-{batch_end}...")

    try:
        bullets_response = fetch_bullets_for_entities(batch)
    except requests.RequestException as exc:
        print(f"Bullet retrieval failed for entities {batch_start}-{batch_end}: {exc}")
        traceback.print_exc()
        all_bullets_responses.append({
            "batch_number": batch_number,
            "batch_start": batch_start,
            "batch_end": batch_end,
            "entity_ids": batch,
            "status": "failed",
            "error": str(exc),
        })
        continue

    batch_results = bullets_response.get("results", []) or []
    all_bullets_results.extend(batch_results)
    all_bullets_responses.append({
        "batch_number": batch_number,
        "batch_start": batch_start,
        "batch_end": batch_end,
        "entity_ids": batch,
        "status": "completed",
        "total_entities": bullets_response.get("total_entities", len(batch_results)),
        "total_bullets": bullets_response.get("total_bullets", 0),
        "response": bullets_response,
    })
    print(f"Retrieved {len(batch_results)} entities and {bullets_response.get('total_bullets', 0)} bullets.")

all_bullets_payload: JsonDict = {
    "retrieved_at_local": datetime.now().isoformat(),
    "batch_size": BATCH_SIZE,
    "total_entities": len(all_bullets_results),
    "total_bullets": sum(int(result.get("total_bullets") or 0) for result in all_bullets_results),
    "results": all_bullets_results,
    "batch_responses": all_bullets_responses,
}

save_json(BRIEF_V2_ALL_BULLETS_FILE, all_bullets_payload)

df_out = flatten_v2_results(all_bullets_results)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="Briefs V2")
    worksheet = writer.sheets["Briefs V2"]
    worksheet.freeze_panes(1, 0)
    worksheet.autofilter(0, 0, max(len(df_out), 1), max(len(df_out.columns) - 1, 0))

    for column_index, column_name in enumerate(df_out.columns):
        max_width = max(len(str(column_name)), 12)
        if not df_out.empty:
            sample_width = df_out[column_name].astype(str).str.slice(0, 80).map(len).max()
            max_width = max(max_width, int(sample_width))
        worksheet.set_column(column_index, column_index, min(max_width + 2, 80))

print(f"Saved raw V2 bullet payload to {BRIEF_V2_ALL_BULLETS_FILE}")
print(f"Written {len(df_out)} rows to {OUT_XLSX}")
display(df_out.head(20))

Retrieving bullets for entities 1-50...
Retrieved 50 entities and 301 bullets.
Retrieving bullets for entities 51-100...
Retrieved 50 entities and 290 bullets.
Retrieving bullets for entities 101-107...
Retrieved 7 entities and 40 bullets.
Saved raw V2 bullet payload to output/brief_v2_all_bullets.json
Written 632 rows to output/portfolio_briefs_generation_v2.xlsx


,rp_entity_id,entity_name,found,total_runs,total_bullets,run_id,report_window_start,report_window_end,run_created_at,run_bullet_count,...,trace_id,bullet_text,embedding_decision,search_action,not_fully_novel,citation_count,citation_ids,citation_headlines,citation_sources,citation_texts
0,B8EF97,Costco Wholesale Corp.,True,1,5,3e5d177b-931c-481e-88b2-f8887cf9f491,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:32.236752,5,...,00f1efa9-7cd9-4b14-9fda-866ecf94b09d,Costco Wholesale Corp. reported fiscal Q2 2026...,keep,keep,False,4,CQS:B896293465F647C1D49FADE6E563860B-1; CQS:A1...,Costco Wholesale Q2 EPS $4.58 Beats $4.57 Esti...,; ; ;,Costco Wholesale (NASDAQ:COST) reported quarte...
1,B8EF97,Costco Wholesale Corp.,True,1,5,3e5d177b-931c-481e-88b2-f8887cf9f491,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:32.236752,5,...,b54e4393-e03c-4426-a399-cd46f3370ccf,Costco Wholesale Corp. posted Q2 2026 net prof...,keep,keep,False,5,CQS:59E907E6750460A40A88EF94875D6199-1; CQS:19...,Costco Wholesale Corp. Reports Q2 Net Profit R...,; ; ; ;,Costco Wholesale Corp. Reports Q2 Net Profit R...
2,B8EF97,Costco Wholesale Corp.,True,1,5,3e5d177b-931c-481e-88b2-f8887cf9f491,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:32.236752,5,...,f8c76606-d04f-43bc-8963-64629d706247,"Costco Wholesale Corp., which told analysts th...",keep,rewrite,False,5,CQS:0574BEC5102B920154320AF4EF9C6F85-2; CQS:50...,Costco Faces Class Action Over Tariff Refunds ...,; ; ; ;,The complaint alleges the lawsuit aims to stop...
3,B8EF97,Costco Wholesale Corp.,True,1,5,3e5d177b-931c-481e-88b2-f8887cf9f491,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:32.236752,5,...,c12033d0-b57b-40d1-8777-24fdcfc595e1,Costco Wholesale Corp. is leveraging its Kirkl...,keep,keep,False,1,CQS:0F7720D811DC1336ACF5AE903868CD2B-1,Costco Tests Private-Label Entry in Energy Dri...,,This article first appeared on GuruFocus.\nCos...
4,B8EF97,Costco Wholesale Corp.,True,1,5,3e5d177b-931c-481e-88b2-f8887cf9f491,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:32.236752,5,...,74056eaa-bbb5-41a3-bab6-92b77d7a3863,Costco Wholesale Corp. continues to monitor th...,keep,keep,False,2,CQS:B977CFD2A5F96001A2C4C329C47560A2-14; CQS:B...,EXTRA: Costco details warehouse growth plans a...,;,"In the warehouses, Vachris said Costco is achi..."
5,BB07E4,Amphenol Corp.,True,1,3,b582e108-6ee6-4e16-b771-314b3c34685c,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:10:59.953560,3,...,259c4d10-de00-461f-8b08-da4ec85df3a6,Amphenol Corporation's Board of Directors appr...,keep,keep,False,3,CQS:C1DBA628A2FFCB09F677A784C7BAD7F6-1; CQS:CB...,Amphenol Announces First Quarter 2026 Dividend...,; ;,Amphenol Corporation (NYSE:APH) announced toda...
6,BB07E4,Amphenol Corp.,True,1,3,b582e108-6ee6-4e16-b771-314b3c34685c,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:10:59.953560,3,...,735474de-2d5d-4d94-b56f-66eff29509e7,Amphenol Corporation introduced the Fiber Syst...,keep,keep,False,2,CQS:E8D341043BE47E274D19EE7B8A423576-1; CQS:E8...,Interstate Connecting Components (ICC) Announc...,;,"LUMBERTON, N.J., March 19, 2026 (GLOBE NEWSWIR..."
7,BB07E4,Amphenol Corp.,True,1,3,b582e108-6ee6-4e16-b771-314b3c34685c,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:10:59.953560,3,...,93e2acec-6282-464b-82fa-b0b1e8f01ac3,Amphenol Corporation's presence at DesignCon 2...,keep,keep,False,1,CQS:1B0F63C6A71ABBA1CB81F01B9EE52534-5,Should Amphenol's Record-Beating Q4 and Higher...,,"In this context, Amphenol's presence at Design..."
8,3461CF,Moody's Corp.,True,1,6,15b68730-5e07-4498-869e-3c8d04852628,2026-02-01T00:00:00,2026-03-30T23:59:59,2026-04-25T21:11:26.946631,6,...,369eb030-1b24-4b2b-ab69-2c291fc658b6,Moody's Corp. communicated that its 2026 opera...,keep,keep,False,1,CQS:433931301B7F521C84C43A412EB4FF56-1,Moody's sees FY26 operating expenses up mid-si...,,"The company said, ""Operating expenses projecte..."
9,3461CF,Moody's Corp.,True,1,6,15b68730-5e07-4498-869e-3c8d04852628,2026-02-0